# ***TEKNOFEST Sağlıkta Yapay Zeka Yarışması***

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
import shap

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
!pip install catboost shap -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
import shap

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Kol 1 - Tabular + CatBoost
### Feature Engineering

In [ ]:
class FeatureEngineer:
    def __init__(self):
        self.median_ = None
        self.iqr_    = None

    def fit_robust_scaler(self, df, num_cols):
        self.median_   = df[num_cols].median()
        q75, q25       = df[num_cols].quantile(0.75), df[num_cols].quantile(0.25)
        self.iqr_      = (q75 - q25).replace(0, 1)
        self.num_cols_ = num_cols

    def apply_robust_scaler(self, df):
        df = df.copy()
        df[self.num_cols_] = (df[self.num_cols_] - self.median_) / self.iqr_
        return df

    def add_inconsistency_score(self, df, clinsig_cols=None):
        df = df.copy()
        if not clinsig_cols:
            return df
        def _score(row):
            vals = [row[c] for c in clinsig_cols if pd.notna(row[c])]
            return round(len(set(vals)) / len(vals), 3) if vals else 0.0
        df['inconsistency_score'] = df.apply(_score, axis=1)
        return df

    def fit_transform(self, df, num_cols, clinsig_cols=None):
        df = self.add_inconsistency_score(df, clinsig_cols)
        self.fit_robust_scaler(df, num_cols)
        return self.apply_robust_scaler(df)

    def transform(self, df, clinsig_cols=None):
        df = self.add_inconsistency_score(df, clinsig_cols)
        return self.apply_robust_scaler(df)


fe = FeatureEngineer()
print('FeatureEngineer hazir.')

### Panel Routing & Embedding

In [ ]:
PANELS = {'genel': 0, 'kanser': 1, 'pah': 2, 'cftr': 3}

PANEL_GENES = {
    'kanser': {'BRCA1','BRCA2','TP53','MLH1','MSH2','APC','PALB2'},
    'pah':    {'BMPR2','ENG','ACVRL1','SMAD9','CAV1'},
    'cftr':   {'CFTR'},
}

def assign_panel(gene: str) -> str:
    for panel, genes in PANEL_GENES.items():
        if gene.upper() in genes:
            return panel
    return 'genel'


class PanelRouter(nn.Module):
    def __init__(self, num_panels=4, embed_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(num_panels, embed_dim)
        self.embed_dim = embed_dim

    def forward(self, panel_ids: torch.Tensor) -> torch.Tensor:
        return self.embedding(panel_ids)   # (B, embed_dim)

    def gene_to_embedding(self, genes: list) -> torch.Tensor:
        ids = torch.tensor([PANELS[assign_panel(g)] for g in genes], dtype=torch.long)
        return self.forward(ids)


router = PanelRouter(num_panels=4, embed_dim=8).to(DEVICE)
test_genes = ['BRCA1', 'CFTR', 'TP53', 'UNKNOWNGENE']
emb = router.gene_to_embedding(test_genes)
print('Panel embedding shape:', emb.shape)   # (4, 8)
print('Panel routing hazir.')

### Expert 1 - CatBoost

In [ ]:
class CatBoostExpert:
    def __init__(self, iterations=1000, depth=6, learning_rate=0.05, random_seed=42):
        self.model = CatBoostClassifier(
            iterations            = iterations,
            depth                 = depth,
            learning_rate         = learning_rate,
            loss_function         = 'Logloss',
            eval_metric           = 'AUC',
            boosting_type         = 'Ordered',       # Ordered boosting
            grow_policy           = 'SymmetricTree', # Symmetric trees
            random_seed           = random_seed,
            verbose               = 100,
            use_best_model        = True,
            early_stopping_rounds = 50,
        )

    def fit(self, X_train, y_train, X_val, y_val, cat_features=None):
        self.model.fit(X_train, y_train,
                       eval_set=(X_val, y_val),
                       cat_features=cat_features or [])
        return self

    def predict_proba(self, X) -> np.ndarray:
        return self.model.predict_proba(X)[:, 1]   # class 1 prob, shape (N,)

    def predict_proba_tensor(self, X) -> torch.Tensor:
        proba = self.predict_proba(X)
        return torch.tensor(proba, dtype=torch.float32).unsqueeze(1)  # (N,1)

    def get_shap_values(self, X) -> np.ndarray:
        return shap.TreeExplainer(self.model).shap_values(X)


expert1 = CatBoostExpert()
print('CatBoost Expert hazir.')
print('Params:', expert1.model.get_params())

## Kol 2 - Sekans + 1D-CNN

In [ ]:
AA_VOCAB    = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX   = {aa: i+1 for i, aa in enumerate(AA_VOCAB)}
AA_TO_IDX['X'] = 0   # bilinmeyen -> PAD
VOCAB_SIZE  = len(AA_VOCAB) + 1  # 21
WINDOW_SIZE = 11  # merkez mutasyon +/- 5 residue


def sequence_to_window(sequence: str, mut_pos: int, window=WINDOW_SIZE) -> list:
    half   = window // 2
    tokens = []
    for i in range(mut_pos - half, mut_pos + half + 1):
        if 0 <= i < len(sequence):
            tokens.append(AA_TO_IDX.get(sequence[i].upper(), 0))
        else:
            tokens.append(0)  # PAD
    return tokens


def batch_sequences(sequences, mut_positions, window=WINDOW_SIZE) -> torch.Tensor:
    windows = [sequence_to_window(s, p, window) for s, p in zip(sequences, mut_positions)]
    return torch.tensor(windows, dtype=torch.long)


class AminoAcidEmbedding(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, padding_idx=0):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.embed_dim = embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embed(x)   # (B, window, embed_dim)


seq_emb = AminoAcidEmbedding().to(DEVICE)
dummy   = batch_sequences(['ACDEFGHIKLM', 'NQRSTVWYACG'], [5, 5]).to(DEVICE)
print('Token shape    :', dummy.shape)           # (2, 11)
print('Embedding shape:', seq_emb(dummy).shape)  # (2, 11, 32)
print('AA Embedding hazir.')

In [ ]:
class CNNExpert(nn.Module):
    '''
    Embedding -> Conv1d(k=3,f=32)+ReLU -> BatchNorm
              -> Conv1d(k=3,f=16)+ReLU -> GlobalMaxPool
              -> Linear(16->8)
    '''
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, out_dim=8):
        super().__init__()
        self.embedding = AminoAcidEmbedding(vocab_size, embed_dim)
        self.conv1     = nn.Conv1d(embed_dim, 32, kernel_size=3, padding=1)
        self.bn1       = nn.BatchNorm1d(32)
        self.conv2     = nn.Conv1d(32, 16, kernel_size=3, padding=1)
        self.fc        = nn.Linear(16, out_dim)
        self.out_dim   = out_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e = self.embedding(x).transpose(1, 2)  # (B, embed_dim, W)
        e = self.bn1(F.relu(self.conv1(e)))    # (B, 32, W)
        e = F.relu(self.conv2(e))              # (B, 16, W)
        e = e.max(dim=2).values                # (B, 16) GlobalMaxPool
        return self.fc(e)                      # (B, out_dim)


cnn_expert = CNNExpert().to(DEVICE)
dummy = batch_sequences(['ACDEFGHIKLM']*4, [5]*4).to(DEVICE)
print('CNN Expert output:', cnn_expert(dummy).shape)  # (4, 8)
print('CNN Expert hazir.')

## Birlestirme + Cikis
### MoE Gating + Concatenation

In [ ]:
class MoEGating(nn.Module):
    '''
    Girisler:
      expert1_prob : (B, 1)  - CatBoost olasiligi
      expert2_feat : (B, 8)  - CNN ozellikleri
      panel_embed  : (B, 8)  - Panel embedding
    Cikis: (B, hidden_dim)
    '''
    def __init__(self, expert1_dim=1, expert2_dim=8, panel_dim=8, hidden_dim=32):
        super().__init__()
        concat_dim  = expert1_dim + expert2_dim + panel_dim   # 17
        self.gate   = nn.Sequential(nn.Linear(concat_dim, 3), nn.Softmax(dim=-1))
        self.proj   = nn.Sequential(
            nn.Linear(concat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )

    def forward(self, expert1_prob, expert2_feat, panel_embed):
        concat = torch.cat([expert1_prob, expert2_feat, panel_embed], dim=1)  # (B,17)
        gates  = self.gate(concat)                                             # (B,3)
        e1_w   = expert1_prob * gates[:, 0:1]
        e2_w   = expert2_feat * gates[:, 1:2]
        ep_w   = panel_embed  * gates[:, 2:3]
        return self.proj(torch.cat([e1_w, e2_w, ep_w], dim=1))                # (B,32)


moe = MoEGating().to(DEVICE)
print('MoE output:', moe(torch.rand(4,1,device=DEVICE),
                         torch.rand(4,8,device=DEVICE),
                         torch.rand(4,8,device=DEVICE)).shape)  # (4,32)
print('MoE Gating hazir.')

In [ ]:
class OutputHead(nn.Module):
    def __init__(self, hidden_dim=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)   # (B, 1)


out_head = OutputHead().to(DEVICE)
print('Output shape:', out_head(torch.rand(4,32,device=DEVICE)).shape)  # (4,1)
print('Output Head hazir.')

### Probability Calibration & Threshold Optimization

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
import warnings; warnings.filterwarnings('ignore')


class ProbabilityCalibrator:
    '''Platt scaling veya Isotonic regression ile kalibrasyon.'''
    def __init__(self, method='isotonic'):
        assert method in ('platt', 'isotonic')
        self.method     = method
        self.calibrator = (IsotonicRegression(out_of_bounds='clip')
                           if method == 'isotonic' else LogisticRegression())

    def fit(self, raw_probs, y_true):
        if self.method == 'platt':
            self.calibrator.fit(raw_probs.reshape(-1, 1), y_true)
        else:
            self.calibrator.fit(raw_probs, y_true)
        return self

    def transform(self, raw_probs):
        if self.method == 'platt':
            return self.calibrator.predict_proba(raw_probs.reshape(-1, 1))[:, 1]
        return self.calibrator.predict(raw_probs)

    def fit_transform(self, raw_probs, y_true):
        return self.fit(raw_probs, y_true).transform(raw_probs)


class ThresholdOptimizer:
    '''Recall >= target_recall kosulunu saglayan en yuksek precision esigi.'''
    def __init__(self, target_recall=1.0):
        self.target_recall = target_recall
        self.threshold_    = 0.5

    def fit(self, probs, y_true):
        best_thresh, best_prec = 0.0, 0.0
        for t in np.arange(0.01, 1.0, 0.01):
            preds = (probs >= t).astype(int)
            if recall_score(y_true, preds, zero_division=0) >= self.target_recall:
                pos  = preds.sum()
                prec = (preds * y_true).sum() / pos if pos > 0 else 0.0
                if prec >= best_prec:
                    best_prec, best_thresh = prec, t
        self.threshold_ = best_thresh if best_thresh > 0 else 0.01
        return self

    def predict(self, probs):
        return (probs >= self.threshold_).astype(int)

    def info(self):
        print(f'Optimum threshold : {self.threshold_:.2f}')
        print(f'Hedef recall      : {self.target_recall:.0%}')


np.random.seed(42)
p = np.random.rand(100)
y = (p > 0.4).astype(int)
cal = ProbabilityCalibrator('isotonic').fit_transform(p, y)
opt = ThresholdOptimizer(1.0).fit(cal, y)
opt.info()
print('Calibration & Threshold hazir.')

### XAI - TreeSHAP & DeepSHAP

In [ ]:
class XAIAnalyzer:
    '''TreeSHAP (CatBoost) + DeepSHAP (1D-CNN) aciklamalari.'''
    def __init__(self, catboost_expert, cnn_expert):
        self.cb_expert  = catboost_expert
        self.cnn_expert = cnn_expert

    def tree_shap(self, X_tabular) -> np.ndarray:
        return shap.TreeExplainer(self.cb_expert.model).shap_values(X_tabular)

    def feature_importance_summary(self, X_tabular, feature_names=None) -> pd.DataFrame:
        shap_vals  = self.tree_shap(X_tabular)
        importance = np.abs(shap_vals).mean(axis=0)
        cols = feature_names or [f'feat_{i}' for i in range(len(importance))]
        return (pd.DataFrame({'feature': cols, 'importance': importance})
                .sort_values('importance', ascending=False)
                .reset_index(drop=True))

    def plot_tree_shap(self, X_tabular, feature_names=None):
        shap.summary_plot(self.tree_shap(X_tabular), X_tabular,
                          feature_names=feature_names, show=True)

    def deep_shap(self, token_batch: torch.Tensor, background: torch.Tensor):
        self.cnn_expert.eval()
        explainer = shap.DeepExplainer(self.cnn_expert, background.to(DEVICE))
        return explainer.shap_values(token_batch.to(DEVICE))


print('XAI Analyzer hazir.')

## Tam Model - TeknoFestModel

In [ ]:
class TeknoFestModel(nn.Module):
    '''
    Kol 1 : FeatureEngineer -> CatBoostExpert -> prob (B,1)
    Kol 2 : sequences       -> CNNExpert      -> feat (B,8)
    Panel : gene_list       -> PanelRouter    -> emb  (B,8)
    Fusion: MoEGating -> OutputHead           -> prob_final (B,1)
    '''
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32,
                 cnn_out_dim=8, panel_dim=8, hidden_dim=32, dropout=0.3):
        super().__init__()
        self.cnn_expert      = CNNExpert(vocab_size, embed_dim, cnn_out_dim)
        self.panel_router    = PanelRouter(num_panels=4, embed_dim=panel_dim)
        self.moe             = MoEGating(1, cnn_out_dim, panel_dim, hidden_dim)
        self.output_head     = OutputHead(hidden_dim, dropout)
        self.catboost_expert = None   # fit sonrasi inject edilir

    def set_catboost_expert(self, expert):
        self.catboost_expert = expert

    def forward(self, token_seq, panel_ids, cb_probs):
        cnn_feat  = self.cnn_expert(token_seq)           # (B, 8)
        panel_emb = self.panel_router(panel_ids)         # (B, 8)
        fused     = self.moe(cb_probs, cnn_feat, panel_emb)  # (B, 32)
        return self.output_head(fused)                   # (B, 1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = TeknoFestModel().to(DEVICE)
print(model)

B            = 4
dummy_tokens = batch_sequences(['ACDEFGHIKLM']*B, [5]*B).to(DEVICE)
dummy_panels = torch.tensor([0, 1, 2, 3], device=DEVICE)
dummy_cb     = torch.rand(B, 1, device=DEVICE)

output = model(dummy_tokens, dummy_panels, dummy_cb)
print(f'Egitilebilir parametre: {model.count_parameters():,}')
print('Model output shape:', output.shape)        # (4, 1)
print('Model output:', output.detach().cpu().numpy().flatten())
print('TeknoFestModel hazir!')

## Egitim Dongusu

In [ ]:
from torch.utils.data import Dataset, DataLoader


class VariantDataset(Dataset):
    '''
    Beklenen df sutunlari:
      sequence : protein sekans (str)
      mut_pos  : mutasyon pozisyonu (int, 0-indexed)
      gene     : gen adi (str)
      label    : 0=benign, 1=patojenik
    cb_probs: CatBoost onceden hesaplanmis olasiliklar (N,)
    '''
    def __init__(self, df, tabular_cols, cb_probs=None):
        self.df           = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.cb_probs     = cb_probs

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        tokens   = torch.tensor(sequence_to_window(row['sequence'], int(row['mut_pos'])), dtype=torch.long)
        panel_id = torch.tensor(PANELS[assign_panel(row['gene'])], dtype=torch.long)
        cb_prob  = torch.tensor([self.cb_probs[idx] if self.cb_probs is not None else 0.5], dtype=torch.float32)
        label    = torch.tensor(row['label'], dtype=torch.float32)
        return tokens, panel_id, cb_prob, label


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for tokens, panel_ids, cb_probs, labels in loader:
        tokens, panel_ids = tokens.to(device), panel_ids.to(device)
        cb_probs, labels  = cb_probs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(tokens, panel_ids, cb_probs).squeeze(1), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    for tokens, panel_ids, cb_probs, labels in loader:
        tokens, panel_ids = tokens.to(device), panel_ids.to(device)
        cb_probs, labels  = cb_probs.to(device), labels.to(device)
        preds = model(tokens, panel_ids, cb_probs).squeeze(1)
        total_loss += criterion(preds, labels).item()
        all_probs.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    return total_loss/len(loader), np.concatenate(all_probs), np.concatenate(all_labels)


def train(model, train_df, val_df, tabular_cols,
          cb_probs_train, cb_probs_val,
          epochs=50, batch_size=32, lr=1e-3,
          pos_weight=None, device=DEVICE):

    train_loader = DataLoader(VariantDataset(train_df, tabular_cols, cb_probs_train),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(VariantDataset(val_df, tabular_cols, cb_probs_val),
                              batch_size=batch_size)

    w         = torch.tensor([pos_weight], device=device) if pos_weight else None
    criterion = nn.BCELoss(weight=w)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    history       = {'train_loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_probs, vl_labels = eval_epoch(model, val_loader, criterion, device)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)

        if epoch % 5 == 0 or epoch == 1:
            recall = recall_score(vl_labels, (vl_probs > 0.5).astype(int), zero_division=0)
            print(f'Epoch {epoch:3d} | Train: {tr_loss:.4f} | Val: {vl_loss:.4f} | Recall: {recall:.3f}')

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), 'teknofest_best.pt')

    print(f'En iyi val loss: {best_val_loss:.4f} -> teknofest_best.pt')
    return history


print('Egitim altyapisi hazir.')
print('Veri gelince: history = train(model, train_df, val_df, tabular_cols, cb_train, cb_val)')

## Pre-Training & Checkpoint Yönetimi

In [ ]:
import os, json as _json, pickle
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score


class CheckpointManager:
    '''
    Kaydeder: PyTorch state, CatBoost .cbm, kalibrator, history JSON, meta JSON.
    Destekler: best checkpoint, periyodik (her N epoch), resume.
    '''
    def __init__(self, save_dir='checkpoints', save_every=10):
        self.save_dir      = save_dir
        self.save_every    = save_every
        self.best_val_loss = float('inf')
        self.history = {'train_loss':[],'val_loss':[],'val_auc':[],'val_prauc':[],'val_f1':[]}
        os.makedirs(save_dir, exist_ok=True)

    def _ckpt_dir(self, tag):
        d = os.path.join(self.save_dir, tag)
        os.makedirs(d, exist_ok=True)
        return d

    def save(self, epoch, model, optimizer, scheduler,
             val_loss, cb_expert=None, calibrator=None, tag=None):
        is_best  = val_loss < self.best_val_loss
        is_per   = (epoch % self.save_every == 0)
        if not (is_best or is_per):
            return is_best
        if is_best:
            self.best_val_loss = val_loss
            tag = 'best'
        if tag is None:
            tag = 'epoch_%04d' % epoch
        d = self._ckpt_dir(tag)
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'best_val_loss': self.best_val_loss}, os.path.join(d, 'pytorch.pt'))
        if cb_expert is not None:
            cb_expert.model.save_model(os.path.join(d, 'catboost.cbm'))
        if calibrator is not None:
            with open(os.path.join(d, 'calibrator.pkl'), 'wb') as f: pickle.dump(calibrator, f)
        with open(os.path.join(d, 'history.json'), 'w') as f: _json.dump(self.history, f, indent=2)
        _json.dump({'epoch':epoch,'val_loss':val_loss,'best_val_loss':self.best_val_loss,'tag':tag},
                   open(os.path.join(d, 'meta.json'), 'w'))
        print('  [ckpt] -> %s  (best=%s)' % (d, is_best))
        return is_best

    def load(self, model, optimizer=None, scheduler=None,
             tag='best', cb_expert=None, calibrator_path=None):
        pt = os.path.join(self.save_dir, tag, 'pytorch.pt')
        if not os.path.exists(pt):
            print('Checkpoint bulunamadi:', pt); return 0
        ckpt = torch.load(pt, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        if optimizer is not None: optimizer.load_state_dict(ckpt['optimizer'])
        if scheduler is not None: scheduler.load_state_dict(ckpt['scheduler'])
        self.best_val_loss = ckpt.get('best_val_loss', float('inf'))
        if cb_expert is not None:
            cb_p = os.path.join(self.save_dir, tag, 'catboost.cbm')
            if os.path.exists(cb_p): cb_expert.model.load_model(cb_p)
        if calibrator_path and os.path.exists(calibrator_path):
            with open(calibrator_path, 'rb') as f: cal = pickle.load(f)
            return ckpt['epoch'], cal
        print('  [ckpt] Yuklendi: epoch=%d  best_loss=%.4f' % (ckpt['epoch'], self.best_val_loss))
        return ckpt['epoch']

    def update_history(self, tr_loss, vl_loss, vl_probs, vl_labels):
        self.history['train_loss'].append(round(tr_loss, 5))
        self.history['val_loss'].append(round(vl_loss, 5))
        try:
            self.history['val_auc'].append(round(roc_auc_score(vl_labels, vl_probs), 4))
            self.history['val_prauc'].append(round(average_precision_score(vl_labels, vl_probs), 4))
            preds = (vl_probs > 0.5).astype(int)
            self.history['val_f1'].append(round(f1_score(vl_labels, preds, zero_division=0), 4))
        except Exception: pass

    def last_metrics(self):
        h = self.history
        if not h['val_auc']: return
        print('  AUC=%.4f | PR-AUC=%.4f | F1=%.4f' % (h['val_auc'][-1], h['val_prauc'][-1], h['val_f1'][-1]))


print('CheckpointManager hazir.')

### Pre-Training Döngüsü (11k Custom Dataset)

In [ ]:
def pretrain(model, train_df, val_df, tabular_cols,
             cb_probs_train, cb_probs_val,
             epochs=100, batch_size=64, lr=1e-3,
             save_dir='checkpoints_pretrain',
             save_every=10, resume=False, device=DEVICE):
    '''11k custom dataset ile pre-training. Resume + periyodik checkpoint destekler.'''
    pin = (device.type == 'cuda')
    tr_ld = DataLoader(VariantDataset(train_df, tabular_cols, cb_probs_train),
                       batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=pin)
    vl_ld = DataLoader(VariantDataset(val_df,   tabular_cols, cb_probs_val),
                       batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    crit  = nn.BCELoss()
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ckpt  = CheckpointManager(save_dir=save_dir, save_every=save_every)
    ep0   = 1
    if resume:
        best_pt = os.path.join(save_dir, 'best', 'pytorch.pt')
        if os.path.exists(best_pt):
            ep0 = ckpt.load(model, opt, sched, tag='best') + 1
            print('Resume: epoch %d den devam' % ep0)
    print('Pre-training: ep%d-%d | device=%s' % (ep0, epochs, device))
    print('Train=%d  Val=%d  Batch=%d' % (len(train_df), len(val_df), batch_size))
    print('-' * 55)
    for epoch in range(ep0, epochs + 1):
        model.train(); tr_loss = 0.0
        for toks, pids, cbp, lbs in tr_ld:
            toks = toks.to(device); pids = pids.to(device)
            cbp  = cbp.to(device);  lbs  = lbs.to(device)
            opt.zero_grad()
            loss = crit(model(toks, pids, cbp).squeeze(1), lbs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); tr_loss += loss.item()
        tr_loss /= len(tr_ld)
        model.eval(); vl_loss = 0.0; all_probs = []; all_labels = []
        with torch.no_grad():
            for toks, pids, cbp, lbs in vl_ld:
                toks = toks.to(device); pids = pids.to(device)
                cbp  = cbp.to(device);  lbs  = lbs.to(device)
                preds = model(toks, pids, cbp).squeeze(1)
                vl_loss += crit(preds, lbs).item()
                all_probs.append(preds.cpu().numpy())
                all_labels.append(lbs.cpu().numpy())
        vl_loss  /= len(vl_ld)
        vl_probs  = np.concatenate(all_probs)
        vl_labels = np.concatenate(all_labels)
        sched.step()
        ckpt.update_history(tr_loss, vl_loss, vl_probs, vl_labels)
        if epoch % 5 == 0 or epoch == 1:
            print('Epoch %3d/%d | Train: %.4f | Val: %.4f' % (epoch, epochs, tr_loss, vl_loss), end=' | ')
            ckpt.last_metrics()
        ckpt.save(epoch, model, opt, sched, vl_loss)
    print('-' * 55)
    print('Pre-training bitti. Best val: %.4f -> %s/best/' % (ckpt.best_val_loss, save_dir))
    return ckpt.history


print('Pre-training hazir. Kullanim:')
print('  history = pretrain(model, train_df, val_df, tabular_cols, cb_tr, cb_val)')
print('  history = pretrain(..., resume=True)  # kaldigi yerden devam')